In [1]:
import pandas as pd
#import cv2
from collections import namedtuple
import os
os.chdir('../')


In [2]:
DataIngestionConfig = namedtuple("DataIngestionConfig", [
    "root_dir",
    "source_URL",
    "local_data_file",
    "unzip_dir"
])

In [3]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen = True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    

In [4]:
from src.collision_alert.constants import *
from src.collision_alert.utils import read_yaml, create_directories

In [5]:
import json
KAGGLE_FILE_PATH = Path('kaggle.json')
def read_json(file_path):
    with open(file_path, 'r') as json_file:
        data = json.load(json_file)
    return data
data = read_json(KAGGLE_FILE_PATH)
os.environ["KAGGLE_USERNAME"] = data['username']
os.environ["KAGGLE_KEY"] = data['key']

In [6]:

class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [7]:
"""
import kaggle
from pathlib import Path

# Set your Kaggle API key file path
kaggle.api.authenticate(api_key=os.environ["KAGGLE_KEY"])  # Replace with your Kaggle API key

# Replace 'your_dataset_name' with the Kaggle dataset name
dataset_name = 'ashfakyeafi/road-vehicle-images-dataset'

# Specify the folder where you want to store the dataset
download_path = Path('./datasets')

# Download the dataset using kaggle library
kaggle.api.dataset_download_files(dataset_name, path=download_path, unzip=True)
"""


'\nimport kaggle\nfrom pathlib import Path\n\n# Set your Kaggle API key file path\nkaggle.api.authenticate(api_key=os.environ["KAGGLE_KEY"])  # Replace with your Kaggle API key\n\n# Replace \'your_dataset_name\' with the Kaggle dataset name\ndataset_name = \'ashfakyeafi/road-vehicle-images-dataset\'\n\n# Specify the folder where you want to store the dataset\ndownload_path = Path(\'./datasets\')\n\n# Download the dataset using kaggle library\nkaggle.api.dataset_download_files(dataset_name, path=download_path, unzip=True)\n'

In [8]:
import os
import urllib.request as request
from zipfile import ZipFile

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, _ = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )

    def _get_updated_list_of_files(self, list_of_files):
        return [f for f in list_of_files]

    def _preprocess(self, zf: ZipFile, f: str, working_dir: str):
        target_filepath = os.path.join(working_dir, f)
        if not os.path.exists(target_filepath):
            zf.extract(f, working_dir)

        if os.path.getsize(target_filepath) == 0:
            os.remove(target_filepath)

    def unzip_and_clean(self):
        with ZipFile(file=self.config.local_data_file, mode="r") as zf:
            list_of_files = zf.namelist()
            updated_list_of_files = self._get_updated_list_of_files(list_of_files)
            for f in updated_list_of_files:
                self._preprocess(zf, f, self.config.unzip_dir)


In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    create_directories([config.config.artifacts_root, config.config.data_ingestion.unzip_dir])

    data_ingestion = DataIngestion(config=data_ingestion_config)
    #data_ingestion.download_file()
    data_ingestion.unzip_and_clean()
except FileNotFoundError as file_not_found_error:
    print(f"File not found: {file_not_found_error}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [10]:
os.getcwd()

'p:\\Collision_Alert_System\\Collision-Alert-System'